In [22]:
from datasets import load_dataset, Video
from huggingface_hub import login
import pandas as pd
import cv2
import mediapipe as mp
import numpy as np
from tqdm import tqdm


In [5]:
from datasets import load_dataset

dataset = load_dataset("AI4A-lab/RecruitView")

In [6]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'video', 'duration', 'question_id', 'question', 'video_quality', 'user_no', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism', 'overall_personality', 'interview_score', 'answer_score', 'speaking_skills', 'confidence_score', 'facial_expression', 'overall_performance', 'transcript', 'gemini_summary'],
        num_rows: 2011
    })
})


In [17]:
print(dataset.keys())

dict_keys(['train'])


In [18]:
train_data = dataset['train']

In [20]:
train_data.features

{'id': Value('string'),
 'video': Video(decode=True, stream_index=None, dimension_order='NCHW', num_ffmpeg_threads=1, device='cpu', seek_mode='exact'),
 'duration': Value('string'),
 'question_id': Value('string'),
 'question': Value('string'),
 'video_quality': Value('string'),
 'user_no': Value('string'),
 'openness': Value('float64'),
 'conscientiousness': Value('float64'),
 'extraversion': Value('float64'),
 'agreeableness': Value('float64'),
 'neuroticism': Value('float64'),
 'overall_personality': Value('float64'),
 'interview_score': Value('float64'),
 'answer_score': Value('float64'),
 'speaking_skills': Value('float64'),
 'confidence_score': Value('float64'),
 'facial_expression': Value('float64'),
 'overall_performance': Value('float64'),
 'transcript': Value('string'),
 'gemini_summary': Value('string')}

In [21]:
print(f"Total records: {len(train_data)}")

Total records: 2011


In [22]:
print(train_data['question'][2000])

What motivates you?


In [23]:
df = dataset['train'].to_pandas()

In [24]:
df.head()

,id,video,duration,question_id,question,video_quality,user_no,openness,conscientiousness,extraversion,...,neuroticism,overall_personality,interview_score,answer_score,speaking_skills,confidence_score,facial_expression,overall_performance,transcript,gemini_summary
0,0001,"{'bytes': None, 'path': 'C:/Users/ma983/.cache...",long,1,Introduce yourself,High,147,-0.653086,-0.048688,-0.691034,...,0.189642,-0.028877,-0.922743,-0.802645,-0.769047,-0.361817,-0.816591,-0.456352,"[00:01 - 00:11] Hello everyone, this is Mathur...","1. Overview:\nThe candidate provides a clear, ..."
1,0002,"{'bytes': None, 'path': 'C:/Users/ma983/.cache...",short,1,Introduce yourself,High,82,-0.007390,0.128426,-0.124832,...,0.087769,-0.064224,-0.173570,-0.132352,-0.561689,-0.174044,-0.304263,-0.193319,[00:00 - 00:05] I'm Darshita Singh from Aligar...,"1. Overview:\nThe candidate, Darshita Sainger,..."
2,0003,"{'bytes': None, 'path': 'C:/Users/ma983/.cache...",medium,1,Introduce yourself,High,103,0.221345,0.280619,0.246076,...,-0.187037,0.599916,0.184024,0.223401,0.223328,0.600258,0.268485,0.262246,"[00:01 - 00:08] Hello everyone, I am Faizan J....",1. Overview:\nThe candidate provides a direct ...
3,0004,"{'bytes': None, 'path': 'C:/Users/ma983/.cache...",medium,1,Introduce yourself,High,277,1.176797,0.763668,1.309153,...,-0.179641,1.278162,1.551773,1.161252,1.648815,1.289105,1.509984,1.338616,,1. Overview:\nThe candidate did not provide a ...
4,0005,"{'bytes': None, 'path': 'C:/Users/ma983/.cache...",long,1,Introduce yourself,High,258,-0.803683,-0.548680,-0.777085,...,0.018259,-0.605465,-1.028743,-0.944125,-1.211701,-0.546533,-0.614527,-0.905753,"[00:01 - 00:05] Hi, my name is Utkarsh Rana an...","1. Overview:\nThe candidate, Utkarsh Rana, pro..."


In [25]:
df['gemini_summary'][1]

'1. Overview:\nThe candidate, Darshita Sainger, provides a concise and direct self-introduction. She clearly states her name, hometown, university, and field of study. The content is factually accurate and directly answers the "introduce yourself" prompt. However, the delivery is marked by a very low-energy, monotone vocal quality and a consistently neutral facial expression. While professional and to the point, this approach misses the opportunity to build rapport or convey enthusiasm, which may leave the interviewer questioning her level of engagement or interest in the opportunity.\n\n2. Communication style & dominant traits:\nThe most dominant observable trait is **Low Extraversion** (Introversion). This is evident in her reserved demeanor, minimal facial expression, lack of expressive gestures, and flat, monotone speech. She presents information factually rather than with social energy. A secondary trait is high **Emotional Stability** (low Neuroticism), as she appears calm and co

In [26]:
print("Columns in dataset:", df.columns.tolist())

Columns in dataset: ['id', 'video', 'duration', 'question_id', 'question', 'video_quality', 'user_no', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism', 'overall_personality', 'interview_score', 'answer_score', 'speaking_skills', 'confidence_score', 'facial_expression', 'overall_performance', 'transcript', 'gemini_summary']


In [27]:
df.describe()

,openness,conscientiousness,extraversion,agreeableness,neuroticism,overall_personality,interview_score,answer_score,speaking_skills,confidence_score,facial_expression,overall_performance
count,2.011000e+03,2.011000e+03,2.011000e+03,2.011000e+03,2.011000e+03,2.011000e+03,2.011000e+03,2.011000e+03,2.011000e+03,2.011000e+03,2.011000e+03,2.011000e+03
mean,-5.858129e-10,-1.810283e-09,-8.733420e-10,-2.707772e-09,5.822762e-10,-4.385845e-09,7.469935e-12,1.960056e-10,2.853436e-10,2.639436e-10,2.253832e-10,2.270578e-10
std,1.126621e+00,8.769595e-01,1.098106e+00,1.200348e+00,4.868924e-01,1.203363e+00,1.230107e+00,1.175536e+00,1.277570e+00,1.079704e+00,1.151031e+00,1.200662e+00
min,-7.799628e+00,-6.928705e+00,-7.184409e+00,-8.412302e+00,-2.576639e+00,-7.526180e+00,-7.860317e+00,-1.019905e+01,-9.406297e+00,-7.286060e+00,-7.461446e+00,-9.311059e+00
25%,-5.492694e-01,-4.483626e-01,-5.464184e-01,-5.656886e-01,-3.062534e-01,-6.114667e-01,-5.957018e-01,-6.105685e-01,-5.580228e-01,-5.352311e-01,-5.838348e-01,-5.748997e-01
50%,8.590045e-03,4.257322e-02,4.365189e-02,-1.596086e-02,2.560906e-02,-1.139338e-02,-2.744952e-02,6.046850e-03,1.987945e-02,1.105049e-02,-1.841234e-02,4.342030e-02
75%,5.222476e-01,4.851403e-01,5.439856e-01,5.477878e-01,3.034359e-01,6.169388e-01,5.454144e-01,5.787526e-01,6.158908e-01,5.531176e-01,5.485763e-01,6.014077e-01
max,9.338849e+00,6.617010e+00,8.910198e+00,9.242025e+00,2.225906e+00,9.269542e+00,9.386007e+00,8.722344e+00,7.901472e+00,6.839430e+00,9.255831e+00,8.848990e+00


In [35]:
from moviepy import VideoFileClip


In [34]:
import sys
!{sys.executable} -m pip install moviepy

  Using cached moviepy-2.2.1-py3-none-any.whl (129 kB)
  Using cached imageio-2.37.3-py3-none-any.whl (317 kB)
  Using cached imageio_ffmpeg-0.6.0-py3-none-win_amd64.whl (31.2 MB)
  Using cached proglog-0.1.12-py3-none-any.whl (6.3 kB)



[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
def extract_audio(video_path, output_path):
    video = VideoFileClip(video_path)
    # Extract audio and save it (disabling logs for cleaner output)
    video.audio.write_audiofile(output_path, logger=None)
    video.close()

In [37]:
from datasets import Video

In [38]:
train_data = train_data.cast_column("video", Video(decode=False))

In [39]:
video_metadata = train_data[0]['video']
sample_video_path = video_metadata['path']

In [40]:
print(f"The actual file path is: {sample_video_path}")

The actual file path is: C:/Users/ma983/.cache/huggingface/hub/datasets--AI4A-lab--RecruitView/snapshots/0cfa07ed0a43622f9104592b100d7bf3a25f6140\videos/vid_0001.mp4


In [41]:
audio_output_path = "extracted_audio.wav"

In [42]:
extract_audio(sample_video_path, audio_output_path)
print(f"Audio successfully extracted to {audio_output_path}")

Audio successfully extracted to extracted_audio.wav


In [43]:
print(df[['interview_score','confidence_score','overall_performance']].describe())
print(df['video_quality'].value_counts())

       interview_score  confidence_score  overall_performance
count     2.011000e+03      2.011000e+03         2.011000e+03
mean      7.469935e-12      2.639436e-10         2.270578e-10
std       1.230107e+00      1.079704e+00         1.200662e+00
min      -7.860317e+00     -7.286060e+00        -9.311059e+00
25%      -5.957018e-01     -5.352311e-01        -5.748997e-01
50%      -2.744952e-02      1.105049e-02         4.342030e-02
75%       5.454144e-01      5.531176e-01         6.014077e-01
max       9.386007e+00      6.839430e+00         8.848990e+00
video_quality
High    1675
Low      336
Name: count, dtype: int64


In [44]:
import warnings
warnings.filterwarnings('ignore')

In [46]:
from datasets import load_dataset
from datasets import Video

ds = load_dataset("AI4A-lab/RecruitView")
train_ds = ds['train']

# Disable auto-decode - ab video field dict milega with 'path'/'bytes', decode nahi hoga
train_ds = train_ds.cast_column("video", Video(decode=False))

sample = train_ds[0]
print("Keys:", sample.keys())
print("Video field type:", type(sample['video']))
print("Video field content:", sample['video'])

Keys: dict_keys(['id', 'video', 'duration', 'question_id', 'question', 'video_quality', 'user_no', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism', 'overall_personality', 'interview_score', 'answer_score', 'speaking_skills', 'confidence_score', 'facial_expression', 'overall_performance', 'transcript', 'gemini_summary'])
Video field type: <class 'dict'>
Video field content: {'bytes': None, 'path': 'C:/Users/ma983/.cache/huggingface/hub/datasets--AI4A-lab--RecruitView/snapshots/0cfa07ed0a43622f9104592b100d7bf3a25f6140\\videos/vid_0001.mp4'}


In [47]:
import tempfile

def get_video_path(video_field):
    """Adjust this based on actual format found above."""
    if isinstance(video_field, str):
        return video_field
    elif isinstance(video_field, dict) and 'path' in video_field:
        return video_field['path']
    elif isinstance(video_field, (bytes, bytearray)):
        tmp = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False)
        tmp.write(video_field)
        tmp.close()
        return tmp.name
    else:
        raise ValueError(f"Unknown video field format: {type(video_field)}")

In [52]:
import mediapipe as mp
print(mp.__version__)
print(dir(mp))

0.10.35
['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


In [1]:
import mediapipe as mp
print(mp.__version__)
print('solutions' in dir(mp))

mp_face_mesh = mp.solutions.face_mesh
print("Success!")

0.10.9
True
Success!


In [3]:
%pip install fer

                                              0.0/891.1 kB ? eta -:--:--
     ------------------------------------- 891.1/891.1 kB 28.4 MB/s eta 0:00:00
                                              0.0/350.8 MB ? eta -:--:--
                                             1.0/350.8 MB 21.4 MB/s eta 0:00:17
                                             1.9/350.8 MB 24.4 MB/s eta 0:00:15
                                             2.8/350.8 MB 19.7 MB/s eta 0:00:18
                                             3.3/350.8 MB 18.9 MB/s eta 0:00:19
                                             3.9/350.8 MB 16.7 MB/s eta 0:00:21
                                             4.4/350.8 MB 16.5 MB/s eta 0:00:21
                                             5.2/350.8 MB 15.9 MB/s eta 0:00:22
                                             5.8/350.8 MB 15.5 MB/s eta 0:00:23
                                             6.5/350.8 MB 15.4 MB/s eta 0:00:23
                                             7.1/350.8

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\ma983\\Interview-Performance-Analyzer\\.venv\\Lib\\site-packages\\~-l\\_imaging.cp311-win_amd64.pyd'
Check the permissions.


[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from fer import FER
emotion_detector = FER(mtcnn=False)
print("Success!")

ModuleNotFoundError: No module named 'fer'

In [5]:
import urllib.request

url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
urllib.request.urlretrieve(url, "face_landmarker.task")
print("Model downloaded")

Model downloaded


In [ ]:
# mp_face_mesh = mp.solutions.face_mesh
# face_mesh = mp_face_mesh.FaceMesh(
#     static_image_mode=False,
#     max_num_faces=1,
#     refine_landmarks=True,
#     min_detection_confidence=0.5
# )

# from fer import FER
# emotion_detector = FER(mtcnn=False)  # mtcnn=True is more accurate but slower

ModuleNotFoundError: No module named 'fer'

In [14]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path=model_path)  # absolute path
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=False,
    num_faces=1,
    running_mode=vision.RunningMode.IMAGE
)
detector = vision.FaceLandmarker.create_from_options(options)
print("Detector ready")

RuntimeError: Unable to open file at c:\Users\ma983\Interview-Performance-Analyzer\.venv\Lib\site-packages/c:\Users\ma983\Interview-Performance-Analyzer\notebooks\face_landmarker.task, errno=22

In [11]:
import os

model_path = os.path.abspath("face_landmarker.task")
model_path = model_path.replace("\\", "/")  # backslash ko forward slash mein convert karo
print("Fixed model path:", model_path)

Fixed model path: c:/Users/ma983/Interview-Performance-Analyzer/notebooks/face_landmarker.task


In [13]:
base_options = python.BaseOptions(model_asset_path="face_landmarker.task")

In [ ]:
def estimate_eye_contact(landmarks, frame_w, frame_h):
    """Simple heuristic: iris/eye landmark deviation from frame center.
    Returns 1 if 'looking at camera' (centered), 0 otherwise.
    Threshold-based - refine later if needed."""
    try:
        left_iris = landmarks.landmark[468]  # requires refine_landmarks=True
        x, y = left_iris.x * frame_w, left_iris.y * frame_h
        center_x, center_y = frame_w / 2, frame_h / 2
        dist = np.sqrt((x - center_x)**2 + (y - center_y)**2)
        threshold = frame_w * 0.15  # tune this
        return 1 if dist < threshold else 0
    except (IndexError, AttributeError):
        return None

In [15]:
import cv2
import numpy as np

def convert_frame_to_mp_image(frame):
    """OpenCV BGR frame ko MediaPipe Image object mein convert karo."""
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)


def estimate_eye_contact_v2(face_landmarks, frame_w, frame_h):
    """Gaze-based heuristic using iris position relative to eye corners."""
    try:
        lm = face_landmarks  # list of NormalizedLandmark

        left_iris = lm[468]
        left_inner = lm[133]
        left_outer = lm[33]

        right_iris = lm[473]
        right_inner = lm[362]
        right_outer = lm[263]

        def gaze_ratio(iris, inner, outer):
            eye_width = outer.x - inner.x
            if eye_width == 0:
                return 0.5
            return (iris.x - inner.x) / eye_width

        left_ratio = gaze_ratio(left_iris, left_inner, left_outer)
        right_ratio = gaze_ratio(right_iris, right_inner, right_outer)
        avg_ratio = (left_ratio + right_ratio) / 2

        return 1 if 0.35 <= avg_ratio <= 0.65 else 0
    except (IndexError, AttributeError):
        return None


def get_emotion_proxy_from_blendshapes(blendshapes):
    """Blendshape categories ko emotion-proxy scores mein map karo."""
    scores = {b.category_name: b.score for b in blendshapes}
    return {
        'smile_score': (scores.get('mouthSmileLeft', 0) + scores.get('mouthSmileRight', 0)) / 2,
        'frown_score': (scores.get('browDownLeft', 0) + scores.get('browDownRight', 0)) / 2,
        'eye_openness': 1 - ((scores.get('eyeBlinkLeft', 0) + scores.get('eyeBlinkRight', 0)) / 2),
        'jaw_open': scores.get('jawOpen', 0),
        'brow_raise': (scores.get('browOuterUpLeft', 0) + scores.get('browOuterUpRight', 0)) / 2,
        'mouth_frown': (scores.get('mouthFrownLeft', 0) + scores.get('mouthFrownRight', 0)) / 2,
    }


def extract_face_features(video_path, sample_rate_sec=1):
    """Extract per-frame face/emotion-proxy features using MediaPipe Tasks API."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or np.isnan(fps):
        fps = 25
    frame_interval = max(1, int(fps * sample_rate_sec))

    frame_records = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            h, w = frame.shape[:2]
            mp_image = convert_frame_to_mp_image(frame)

            result = detector.detect(mp_image)

            face_detected = len(result.face_landmarks) > 0
            eye_contact = None
            emotion_proxy = None

            if face_detected:
                landmarks = result.face_landmarks[0]
                eye_contact = estimate_eye_contact_v2(landmarks, w, h)

                if result.face_blendshapes:
                    emotion_proxy = get_emotion_proxy_from_blendshapes(result.face_blendshapes[0])

            frame_records.append({
                'face_detected': face_detected,
                'eye_contact': eye_contact,
                'emotion_proxy': emotion_proxy
            })
        frame_idx += 1

    cap.release()
    return frame_records

In [16]:
EMOTION_PROXY_KEYS = ['smile_score', 'frown_score', 'eye_openness', 'jaw_open', 'brow_raise', 'mouth_frown']

def aggregate_video_features(frame_records, video_id):
    total_frames = len(frame_records)
    if total_frames == 0:
        return {'id': video_id, 'face_detected_ratio': 0.0}

    face_detected_count = sum(1 for r in frame_records if r['face_detected'])
    eye_contact_vals = [r['eye_contact'] for r in frame_records if r['eye_contact'] is not None]

    result = {
        'id': video_id,
        'face_detected_ratio': face_detected_count / total_frames,
        'eye_contact_pct': (sum(eye_contact_vals) / len(eye_contact_vals)) if eye_contact_vals else np.nan,
    }

    for key in EMOTION_PROXY_KEYS:
        vals = [r['emotion_proxy'][key] for r in frame_records if r['emotion_proxy'] is not None]
        result[f'{key}_mean'] = np.mean(vals) if vals else np.nan
        result[f'{key}_std'] = np.std(vals) if vals else np.nan

    return result

In [34]:
sample = train_ds[0]
video_path = get_video_path(sample['video'])
frame_records = extract_face_features(video_path, sample_rate_sec=1)
print(f"Frames processed: {len(frame_records)}")
print(f"Sample record: {frame_records[0]}")

agg = aggregate_video_features(frame_records, sample['id'])
print(agg)

Frames processed: 44
Sample record: {'face_detected': True, 'eye_contact': 1, 'emotion_proxy': {'smile_score': 0.14586132019758224, 'frown_score': 0.0042698223842307925, 'eye_openness': 0.8115199506282806, 'jaw_open': 0.00020569887419696897, 'brow_raise': 0.24518554657697678, 'mouth_frown': 0.0016613111947663128}}
{'id': '0001', 'face_detected_ratio': 1.0, 'eye_contact_pct': 1.0, 'smile_score_mean': np.float64(0.45887441840287385), 'smile_score_std': np.float64(0.25219741502022963), 'frown_score_mean': np.float64(0.0013334394271541069), 'frown_score_std': np.float64(0.001154940649545297), 'eye_openness_mean': np.float64(0.7671516596720639), 'eye_openness_std': np.float64(0.17557922610784069), 'jaw_open_mean': np.float64(0.015858630677526395), 'jaw_open_std': np.float64(0.02211229048848542), 'brow_raise_mean': np.float64(0.39942656321959064), 'brow_raise_std': np.float64(0.1047675455437204), 'mouth_frown_mean': np.float64(0.0017325413910860086), 'mouth_frown_std': np.float64(0.004994546

In [27]:
from datasets import load_dataset, Video

ds = load_dataset("AI4A-lab/RecruitView")
train_ds = ds['train']
train_ds = train_ds.cast_column("video", Video(decode=False))

In [25]:
from datasets import load_dataset, Video

In [29]:
import tempfile

def get_video_path(video_field):
    """Adjust this based on actual format found above."""
    if isinstance(video_field, str):
        return video_field
    elif isinstance(video_field, dict) and 'path' in video_field:
        return video_field['path']
    elif isinstance(video_field, (bytes, bytearray)):
        tmp = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False)
        tmp.write(video_field)
        tmp.close()
        return tmp.name
    else:
        raise ValueError(f"Unknown video field format: {type(video_field)}")

In [31]:
import urllib.request

url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
urllib.request.urlretrieve(url, "face_landmarker.task")
print("Model downloaded")

Model downloaded


In [33]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# File ko bytes mein read karo directly
with open("face_landmarker.task", "rb") as f:
    model_data = f.read()

base_options = python.BaseOptions(model_asset_buffer=model_data)  # path nahi, buffer
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=False,
    num_faces=1,
    running_mode=vision.RunningMode.IMAGE
)
detector = vision.FaceLandmarker.create_from_options(options)
print("Detector ready")

Detector ready


In [35]:
SUBSET_SIZE = 50

subset = train_ds.select(range(SUBSET_SIZE))
results = []
failed_ids = []

for sample in tqdm(subset, desc="Processing videos"):
    video_id = sample['id']
    try:
        video_path = get_video_path(sample['video'])
        frame_records = extract_face_features(video_path, sample_rate_sec=1)
        agg_features = aggregate_video_features(frame_records, video_id)
        results.append(agg_features)
    except Exception as e:
        print(f"Failed on id={video_id}: {e}")
        failed_ids.append(video_id)

print(f"\nProcessed: {len(results)}, Failed: {len(failed_ids)}")

Processing videos: 100%|██████████| 50/50 [01:54<00:00,  2.28s/it]


Processed: 50, Failed: 0


In [36]:
face_df = pd.DataFrame(results)
print(face_df.shape)
print(face_df.isnull().sum())
face_df.head()

(50, 15)
id                     0
face_detected_ratio    0
eye_contact_pct        0
smile_score_mean       0
smile_score_std        0
frown_score_mean       0
frown_score_std        0
eye_openness_mean      0
eye_openness_std       0
jaw_open_mean          0
jaw_open_std           0
brow_raise_mean        0
brow_raise_std         0
mouth_frown_mean       0
mouth_frown_std        0
dtype: int64


,id,face_detected_ratio,eye_contact_pct,smile_score_mean,smile_score_std,frown_score_mean,frown_score_std,eye_openness_mean,eye_openness_std,jaw_open_mean,jaw_open_std,brow_raise_mean,brow_raise_std,mouth_frown_mean,mouth_frown_std
0,0001,1.0,1.0,0.458874,0.252197,0.001333,0.001155,0.767152,0.175579,0.015859,0.022112,0.399427,0.104768,0.001733,0.004995
1,0002,1.0,1.0,0.075525,0.071071,0.000263,0.000241,0.861586,0.172239,0.034392,0.042187,0.631968,0.105312,0.004895,0.007879
2,0003,1.0,1.0,0.043278,0.093646,0.006341,0.005869,0.894525,0.036300,0.019516,0.020920,0.335968,0.110311,0.000369,0.000542
3,0004,1.0,1.0,0.000419,0.001847,0.012285,0.066007,0.920164,0.071501,0.001745,0.001187,0.470537,0.165819,0.010575,0.024723
4,0005,1.0,1.0,0.043369,0.065311,0.026583,0.036279,0.875688,0.128783,0.029904,0.053328,0.102671,0.070685,0.000919,0.002585


In [38]:
# temporarily test on one video's frames, print raw ratios
sample = train_ds[0]
video_path = get_video_path(sample['video'])

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
mp_image = convert_frame_to_mp_image(frame)
result = detector.detect(mp_image)

if result.face_landmarks:
    lm = result.face_landmarks[0]
    left_iris, left_inner, left_outer = lm[468], lm[133], lm[33]
    right_iris, right_inner, right_outer = lm[473], lm[362], lm[263]
    
    left_ratio = (left_iris.x - left_inner.x) / (left_outer.x - left_inner.x)
    right_ratio = (right_iris.x - right_inner.x) / (right_outer.x - right_inner.x)
    print(f"Left ratio: {left_ratio:.3f}, Right ratio: {right_ratio:.3f}")
cap.release()

Left ratio: 0.549, Right ratio: 0.512


In [39]:
sample = train_ds[0]
video_path = get_video_path(sample['video'])

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = max(1, int(fps * 1))  # 1 frame/sec, same as pipeline

ratios = []
frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        mp_image = convert_frame_to_mp_image(frame)
        result = detector.detect(mp_image)
        if result.face_landmarks:
            lm = result.face_landmarks[0]
            left_iris, left_inner, left_outer = lm[468], lm[133], lm[33]
            right_iris, right_inner, right_outer = lm[473], lm[362], lm[263]
            left_ratio = (left_iris.x - left_inner.x) / (left_outer.x - left_inner.x)
            right_ratio = (right_iris.x - right_inner.x) / (right_outer.x - right_inner.x)
            avg_ratio = (left_ratio + right_ratio) / 2
            ratios.append(avg_ratio)
    frame_idx += 1
cap.release()

print(f"Total frames analyzed: {len(ratios)}")
print(f"Min: {min(ratios):.3f}, Max: {max(ratios):.3f}, Mean: {np.mean(ratios):.3f}, Std: {np.std(ratios):.3f}")
print("First 20 ratios:", [round(r, 3) for r in ratios[:20]])

Total frames analyzed: 44
Min: 0.473, Max: 0.555, Mean: 0.526, Std: 0.017
First 20 ratios: [0.531, 0.537, 0.536, 0.53, 0.518, 0.544, 0.537, 0.531, 0.537, 0.533, 0.541, 0.473, 0.54, 0.517, 0.528, 0.528, 0.539, 0.527, 0.521, 0.522]


In [40]:
def estimate_gaze_ratio(face_landmarks):
    """Return raw gaze ratio (continuous), not binary. 
    ~0.5 = centered/camera-facing, deviation = looking away."""
    try:
        lm = face_landmarks
        left_iris, left_inner, left_outer = lm[468], lm[133], lm[33]
        right_iris, right_inner, right_outer = lm[473], lm[362], lm[263]

        def ratio(iris, inner, outer):
            w = outer.x - inner.x
            return (iris.x - inner.x) / w if w != 0 else 0.5

        left_ratio = ratio(left_iris, left_inner, left_outer)
        right_ratio = ratio(right_iris, right_inner, right_outer)
        return (left_ratio + right_ratio) / 2
    except (IndexError, AttributeError):
        return None

In [41]:
gaze_ratio = estimate_gaze_ratio(landmarks)
...
frame_records.append({..., 'gaze_ratio': gaze_ratio, ...})

SyntaxError: invalid syntax (2943805897.py, line 3)

In [42]:
gaze_vals = [r['gaze_ratio'] for r in frame_records if r['gaze_ratio'] is not None]
result['gaze_ratio_mean'] = np.mean(gaze_vals) if gaze_vals else np.nan
result['gaze_deviation_mean'] = np.mean([abs(v - 0.5) for v in gaze_vals]) if gaze_vals else np.nan  # kitna center se hata hua
result['gaze_stability_std'] = np.std(gaze_vals) if gaze_vals else np.nan  # kitna consistent/stable gaze raha

KeyError: 'gaze_ratio'

In [43]:
def extract_face_features(video_path, sample_rate_sec=1):
    """Extract per-frame face/emotion-proxy features using MediaPipe Tasks API."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or np.isnan(fps):
        fps = 25
    frame_interval = max(1, int(fps * sample_rate_sec))

    frame_records = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            h, w = frame.shape[:2]
            mp_image = convert_frame_to_mp_image(frame)

            result = detector.detect(mp_image)

            face_detected = len(result.face_landmarks) > 0
            gaze_ratio = None
            emotion_proxy = None

            if face_detected:
                landmarks = result.face_landmarks[0]
                gaze_ratio = estimate_gaze_ratio(landmarks)

                if result.face_blendshapes:
                    emotion_proxy = get_emotion_proxy_from_blendshapes(result.face_blendshapes[0])

            frame_records.append({
                'face_detected': face_detected,
                'gaze_ratio': gaze_ratio,
                'emotion_proxy': emotion_proxy
            })
        frame_idx += 1

    cap.release()
    return frame_records


In [44]:
EMOTION_PROXY_KEYS = ['smile_score', 'frown_score', 'eye_openness', 'jaw_open', 'brow_raise', 'mouth_frown']

def aggregate_video_features(frame_records, video_id):
    total_frames = len(frame_records)
    if total_frames == 0:
        return {'id': video_id, 'face_detected_ratio': 0.0}

    face_detected_count = sum(1 for r in frame_records if r['face_detected'])
    gaze_vals = [r['gaze_ratio'] for r in frame_records if r['gaze_ratio'] is not None]

    result = {
        'id': video_id,
        'face_detected_ratio': face_detected_count / total_frames,
        'gaze_ratio_mean': np.mean(gaze_vals) if gaze_vals else np.nan,
        'gaze_deviation_mean': np.mean([abs(v - 0.5) for v in gaze_vals]) if gaze_vals else np.nan,
        'gaze_stability_std': np.std(gaze_vals) if gaze_vals else np.nan,
    }

    for key in EMOTION_PROXY_KEYS:
        vals = [r['emotion_proxy'][key] for r in frame_records if r['emotion_proxy'] is not None]
        result[f'{key}_mean'] = np.mean(vals) if vals else np.nan
        result[f'{key}_std'] = np.std(vals) if vals else np.nan

    return result

In [45]:
def estimate_gaze_ratio(face_landmarks):
    """Return raw gaze ratio (continuous), not binary.
    ~0.5 = centered/camera-facing, deviation = looking away."""
    try:
        lm = face_landmarks
        left_iris, left_inner, left_outer = lm[468], lm[133], lm[33]
        right_iris, right_inner, right_outer = lm[473], lm[362], lm[263]

        def ratio(iris, inner, outer):
            w = outer.x - inner.x
            return (iris.x - inner.x) / w if w != 0 else 0.5

        left_ratio = ratio(left_iris, left_inner, left_outer)
        right_ratio = ratio(right_iris, right_inner, right_outer)
        return (left_ratio + right_ratio) / 2
    except (IndexError, AttributeError):
        return None

In [46]:
sample = train_ds[0]
video_path = get_video_path(sample['video'])
frame_records = extract_face_features(video_path, sample_rate_sec=1)
agg = aggregate_video_features(frame_records, sample['id'])
print(agg)

{'id': '0001', 'face_detected_ratio': 1.0, 'gaze_ratio_mean': np.float64(0.5259281626346135), 'gaze_deviation_mean': np.float64(0.02821083486828747), 'gaze_stability_std': np.float64(0.016503204820425815), 'smile_score_mean': np.float64(0.45887441840287385), 'smile_score_std': np.float64(0.25219741502022963), 'frown_score_mean': np.float64(0.0013334394271541069), 'frown_score_std': np.float64(0.001154940649545297), 'eye_openness_mean': np.float64(0.7671516596720639), 'eye_openness_std': np.float64(0.17557922610784069), 'jaw_open_mean': np.float64(0.015858630677526395), 'jaw_open_std': np.float64(0.02211229048848542), 'brow_raise_mean': np.float64(0.39942656321959064), 'brow_raise_std': np.float64(0.1047675455437204), 'mouth_frown_mean': np.float64(0.0017325413910860086), 'mouth_frown_std': np.float64(0.004994546144234892)}


In [47]:
SUBSET_SIZE = 50

subset = train_ds.select(range(SUBSET_SIZE))
results = []
failed_ids = []

for sample in tqdm(subset, desc="Processing videos"):
    video_id = sample['id']
    try:
        video_path = get_video_path(sample['video'])
        frame_records = extract_face_features(video_path, sample_rate_sec=1)
        agg_features = aggregate_video_features(frame_records, video_id)
        results.append(agg_features)
    except Exception as e:
        print(f"Failed on id={video_id}: {e}")
        failed_ids.append(video_id)

print(f"\nProcessed: {len(results)}, Failed: {len(failed_ids)}")

face_df = pd.DataFrame(results)
print(face_df[['gaze_ratio_mean', 'gaze_deviation_mean', 'gaze_stability_std']].describe())


Processing videos: 100%|██████████| 50/50 [01:35<00:00,  1.91s/it]


Processed: 50, Failed: 0
       gaze_ratio_mean  gaze_deviation_mean  gaze_stability_std
count        50.000000            50.000000           50.000000
mean          0.543498             0.046618            0.014929
std           0.021407             0.016573            0.007628
min           0.462644             0.017225            0.000000
25%           0.529914             0.035425            0.010359
50%           0.546971             0.047416            0.014368
75%           0.555389             0.057499            0.019413
max           0.593572             0.093572            0.043424


In [48]:
CHECKPOINT_EVERY = 100
results = []
failed_ids = []

for i, sample in enumerate(tqdm(train_ds, desc="Processing full dataset")):
    video_id = sample['id']
    try:
        video_path = get_video_path(sample['video'])
        frame_records = extract_face_features(video_path, sample_rate_sec=1)
        agg_features = aggregate_video_features(frame_records, video_id)
        results.append(agg_features)
    except Exception as e:
        print(f"Failed on id={video_id}: {e}")
        failed_ids.append(video_id)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv('face_features_checkpoint.csv', index=False)
        print(f"Checkpoint saved at {i+1} videos")

face_df = pd.DataFrame(results)
face_df.to_csv('face_features.csv', index=False)
print("Final save complete:", face_df.shape)
print(f"\nFailed videos: {len(failed_ids)}")
print(failed_ids[:20])  # first 20 failed ids, agar koi hon

Processing full dataset:   5%|▍         | 100/2011 [03:33<1:35:12,  2.99s/it]

Checkpoint saved at 100 videos


Processing full dataset:  10%|▉         | 200/2011 [06:20<50:39,  1.68s/it]  

Checkpoint saved at 200 videos


Processing full dataset:  15%|█▍        | 300/2011 [09:13<1:04:11,  2.25s/it]

...

Checkpoint saved at 1900 videos


Processing full dataset:  99%|█████████▉| 2000/2011 [1:01:11<00:20,  1.90s/it]

Checkpoint saved at 2000 videos


Processing full dataset: 100%|██████████| 2011/2011 [1:01:30<00:00,  1.84s/it]

Final save complete: (2011, 17)

Failed videos: 0
[]


In [49]:
def blendshapes_to_emotions(blendshapes):
    """MediaPipe blendshapes ko 7 standard emotion probabilities mein map karta hai."""
    scores = {b.category_name: b.score for b in blendshapes}
    g = lambda k: scores.get(k, 0.0)

    happy = (g('mouthSmileLeft') + g('mouthSmileRight')) / 2 * (1 - g('jawOpen') * 0.3)
    sad = ((g('mouthFrownLeft') + g('mouthFrownRight')) / 2 * 0.6 +
           g('browInnerUp') * 0.4 +
           (g('mouthLowerDownLeft') + g('mouthLowerDownRight')) / 2 * 0.3)
    angry = ((g('browDownLeft') + g('browDownRight')) / 2 * 0.5 +
             (g('noseSneerLeft') + g('noseSneerRight')) / 2 * 0.3 +
             (g('mouthPressLeft') + g('mouthPressRight')) / 2 * 0.2)
    surprise = ((g('browOuterUpLeft') + g('browOuterUpRight')) / 2 * 0.4 +
                (g('eyeWideLeft') + g('eyeWideRight')) / 2 * 0.3 +
                g('jawOpen') * 0.3)
    fear = (g('browInnerUp') * 0.3 +
            (g('eyeWideLeft') + g('eyeWideRight')) / 2 * 0.4 +
            (g('mouthStretchLeft') + g('mouthStretchRight')) / 2 * 0.3)
    disgust = ((g('noseSneerLeft') + g('noseSneerRight')) / 2 * 0.5 +
               (g('mouthUpperUpLeft') + g('mouthUpperUpRight')) / 2 * 0.3 +
               (g('browDownLeft') + g('browDownRight')) / 2 * 0.2)

    raw_scores = {'happy': happy, 'sad': sad, 'angry': angry,
                  'surprise': surprise, 'fear': fear, 'disgust': disgust}
    total_activation = sum(raw_scores.values())
    raw_scores['neutral'] = max(0.0, 1 - total_activation)

    total = sum(raw_scores.values())
    if total > 0:
        return {k: v / total for k, v in raw_scores.items()}
    return {k: (1.0 if k == 'neutral' else 0.0) for k in raw_scores}

In [50]:
if result.face_blendshapes:
    emotion_proxy = get_emotion_proxy_from_blendshapes(result.face_blendshapes[0])
    emotion_7class = blendshapes_to_emotions(result.face_blendshapes[0])   # ← ye naya add karo

In [51]:
frame_records.append({
    'face_detected': face_detected,
    'gaze_ratio': gaze_ratio,
    'emotion_proxy': emotion_proxy,
    'emotion_7class': emotion_7class    # ← ye naya add karo
})

NameError: name 'face_detected' is not defined

In [52]:
print(face_df.head())
print(face_df.describe())
print(face_df.isnull().sum())
print(face_df.shape)

     id  face_detected_ratio  gaze_ratio_mean  gaze_deviation_mean  \
0  0001                  1.0         0.525928             0.028211   
1  0002                  1.0         0.542930             0.046900   
2  0003                  1.0         0.546025             0.046025   
3  0004                  1.0         0.541986             0.042423   
4  0005                  1.0         0.546883             0.048153   

   gaze_stability_std  smile_score_mean  smile_score_std  frown_score_mean  \
0            0.016503          0.458874         0.252197          0.001333   
1            0.021770          0.075525         0.071071          0.000263   
2            0.008399          0.043278         0.093646          0.006341   
3            0.014829          0.000419         0.001847          0.012285   
4            0.017564          0.043369         0.065311          0.026583   

   frown_score_std  eye_openness_mean  eye_openness_std  jaw_open_mean  \
0         0.001155           0.76715

In [53]:
def blendshapes_to_emotions(blendshapes):
    """MediaPipe blendshapes ko 7 standard emotion probabilities mein map karta hai."""
    scores = {b.category_name: b.score for b in blendshapes}
    g = lambda k: scores.get(k, 0.0)

    happy = (g('mouthSmileLeft') + g('mouthSmileRight')) / 2 * (1 - g('jawOpen') * 0.3)
    sad = ((g('mouthFrownLeft') + g('mouthFrownRight')) / 2 * 0.6 +
           g('browInnerUp') * 0.4 +
           (g('mouthLowerDownLeft') + g('mouthLowerDownRight')) / 2 * 0.3)
    angry = ((g('browDownLeft') + g('browDownRight')) / 2 * 0.5 +
             (g('noseSneerLeft') + g('noseSneerRight')) / 2 * 0.3 +
             (g('mouthPressLeft') + g('mouthPressRight')) / 2 * 0.2)
    surprise = ((g('browOuterUpLeft') + g('browOuterUpRight')) / 2 * 0.4 +
                (g('eyeWideLeft') + g('eyeWideRight')) / 2 * 0.3 +
                g('jawOpen') * 0.3)
    fear = (g('browInnerUp') * 0.3 +
            (g('eyeWideLeft') + g('eyeWideRight')) / 2 * 0.4 +
            (g('mouthStretchLeft') + g('mouthStretchRight')) / 2 * 0.3)
    disgust = ((g('noseSneerLeft') + g('noseSneerRight')) / 2 * 0.5 +
               (g('mouthUpperUpLeft') + g('mouthUpperUpRight')) / 2 * 0.3 +
               (g('browDownLeft') + g('browDownRight')) / 2 * 0.2)

    raw_scores = {'happy': happy, 'sad': sad, 'angry': angry,
                  'surprise': surprise, 'fear': fear, 'disgust': disgust}
    total_activation = sum(raw_scores.values())
    raw_scores['neutral'] = max(0.0, 1 - total_activation)

    total = sum(raw_scores.values())
    if total > 0:
        return {k: v / total for k, v in raw_scores.items()}
    return {k: (1.0 if k == 'neutral' else 0.0) for k in raw_scores}

In [54]:
def extract_face_features(video_path, sample_rate_sec=1):
    """Extract per-frame face/emotion features using MediaPipe Tasks API."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or np.isnan(fps):
        fps = 25
    frame_interval = max(1, int(fps * sample_rate_sec))

    frame_records = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            h, w = frame.shape[:2]
            mp_image = convert_frame_to_mp_image(frame)
            result = detector.detect(mp_image)

            face_detected = len(result.face_landmarks) > 0
            gaze_ratio = None
            emotion_proxy = None
            emotion_7class = None

            if face_detected:
                landmarks = result.face_landmarks[0]
                gaze_ratio = estimate_gaze_ratio(landmarks)

                if result.face_blendshapes:
                    emotion_proxy = get_emotion_proxy_from_blendshapes(result.face_blendshapes[0])
                    emotion_7class = blendshapes_to_emotions(result.face_blendshapes[0])

            frame_records.append({
                'face_detected': face_detected,
                'gaze_ratio': gaze_ratio,
                'emotion_proxy': emotion_proxy,
                'emotion_7class': emotion_7class
            })
        frame_idx += 1

    cap.release()
    return frame_records

In [55]:
EMOTION_PROXY_KEYS = ['smile_score', 'frown_score', 'eye_openness', 'jaw_open', 'brow_raise', 'mouth_frown']
EMOTION_7_LABELS = ['happy', 'sad', 'angry', 'surprise', 'fear', 'disgust', 'neutral']

def aggregate_video_features(frame_records, video_id):
    total_frames = len(frame_records)
    if total_frames == 0:
        return {'id': video_id, 'face_detected_ratio': 0.0}

    face_detected_count = sum(1 for r in frame_records if r['face_detected'])
    gaze_vals = [r['gaze_ratio'] for r in frame_records if r['gaze_ratio'] is not None]

    result = {
        'id': video_id,
        'face_detected_ratio': face_detected_count / total_frames,
        'gaze_ratio_mean': np.mean(gaze_vals) if gaze_vals else np.nan,
        'gaze_deviation_mean': np.mean([abs(v - 0.5) for v in gaze_vals]) if gaze_vals else np.nan,
        'gaze_stability_std': np.std(gaze_vals) if gaze_vals else np.nan,
    }

    for key in EMOTION_PROXY_KEYS:
        vals = [r['emotion_proxy'][key] for r in frame_records if r['emotion_proxy'] is not None]
        result[f'{key}_mean'] = np.mean(vals) if vals else np.nan
        result[f'{key}_std'] = np.std(vals) if vals else np.nan

    for label in EMOTION_7_LABELS:
        vals = [r['emotion_7class'][label] for r in frame_records if r.get('emotion_7class') is not None]
        result[f'emotion_{label}_mean'] = np.mean(vals) if vals else np.nan

    return result

In [56]:
sample = train_ds[0]
video_path = get_video_path(sample['video'])
frame_records = extract_face_features(video_path, sample_rate_sec=1)
agg = aggregate_video_features(frame_records, sample['id'])
print(agg)

{'id': '0001', 'face_detected_ratio': 1.0, 'gaze_ratio_mean': np.float64(0.5259281626346135), 'gaze_deviation_mean': np.float64(0.02821083486828747), 'gaze_stability_std': np.float64(0.016503204820425815), 'smile_score_mean': np.float64(0.45887441840287385), 'smile_score_std': np.float64(0.25219741502022963), 'frown_score_mean': np.float64(0.0013334394271541069), 'frown_score_std': np.float64(0.001154940649545297), 'eye_openness_mean': np.float64(0.7671516596720639), 'eye_openness_std': np.float64(0.17557922610784069), 'jaw_open_mean': np.float64(0.015858630677526395), 'jaw_open_std': np.float64(0.02211229048848542), 'brow_raise_mean': np.float64(0.39942656321959064), 'brow_raise_std': np.float64(0.1047675455437204), 'mouth_frown_mean': np.float64(0.0017325413910860086), 'mouth_frown_std': np.float64(0.004994546144234892), 'emotion_happy_mean': np.float64(0.3860009004103384), 'emotion_sad_mean': np.float64(0.15330118908608165), 'emotion_angry_mean': np.float64(0.007711355055406602), 'e

In [57]:
CHECKPOINT_EVERY = 100
results = []
failed_ids = []

for i, sample in enumerate(tqdm(train_ds, desc="Processing full dataset with emotions")):
    video_id = sample['id']
    try:
        video_path = get_video_path(sample['video'])
        frame_records = extract_face_features(video_path, sample_rate_sec=1)
        agg_features = aggregate_video_features(frame_records, video_id)
        results.append(agg_features)
    except Exception as e:
        print(f"Failed on id={video_id}: {e}")
        failed_ids.append(video_id)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv('face_features_checkpoint.csv', index=False)
        print(f"Checkpoint saved at {i+1} videos")

face_df = pd.DataFrame(results)
face_df.to_csv('face_features_final.csv', index=False)
print("Done:", face_df.shape)

Processing full dataset with emotions:   5%|▍         | 100/2011 [03:26<1:01:08,  1.92s/it]

Checkpoint saved at 100 videos


Processing full dataset with emotions:  10%|▉         | 200/2011 [06:18<51:15,  1.70s/it]  

Checkpoint saved at 200 videos


Processing full dataset with emotions:  15%|█▍        | 300/2011 [09:34<1:10:00,  2.46s/it]

...

Checkpoint saved at 1900 videos


Processing full dataset with emotions:  99%|█████████▉| 2000/2011 [1:18:08<00:25,  2.28s/it]

Checkpoint saved at 2000 videos


Processing full dataset with emotions: 100%|██████████| 2011/2011 [1:18:33<00:00,  2.34s/it]

Done: (2011, 24)


In [58]:
import pandas as pd

In [59]:
df = pd.read_csv("face_features_final.csv")

In [61]:
df.head()

,id,face_detected_ratio,gaze_ratio_mean,gaze_deviation_mean,gaze_stability_std,smile_score_mean,smile_score_std,frown_score_mean,frown_score_std,eye_openness_mean,...,brow_raise_std,mouth_frown_mean,mouth_frown_std,emotion_happy_mean,emotion_sad_mean,emotion_angry_mean,emotion_surprise_mean,emotion_fear_mean,emotion_disgust_mean,emotion_neutral_mean
0,1,1.0,0.525928,0.028211,0.016503,0.458874,0.252197,0.001333,0.001155,0.767152,...,0.104768,0.001733,0.004995,0.386001,0.153301,0.007711,0.147903,0.103187,0.061469,0.140429
1,2,1.0,0.542930,0.046900,0.021770,0.075525,0.071071,0.000263,0.000241,0.861586,...,0.105312,0.004895,0.007879,0.069178,0.264579,0.006462,0.257363,0.208399,0.033720,0.160299
2,3,1.0,0.546025,0.046025,0.008399,0.043278,0.093646,0.006341,0.005869,0.894525,...,0.110311,0.000369,0.000542,0.043146,0.031519,0.016211,0.142088,0.026297,0.017807,0.722932
3,4,1.0,0.541986,0.042423,0.014829,0.000419,0.001847,0.012285,0.066007,0.920164,...,0.165819,0.010575,0.024723,0.000418,0.188984,0.007895,0.193910,0.145331,0.003834,0.459627
4,5,1.0,0.546883,0.048153,0.017564,0.043369,0.065311,0.026583,0.036279,0.875688,...,0.070685,0.000919,0.002585,0.043110,0.106432,0.023733,0.054067,0.065449,0.039123,0.668087


In [62]:
def compute_face_geometry(landmarks, frame_w, frame_h):
    xs = [lm.x * frame_w for lm in landmarks]
    ys = [lm.y * frame_h for lm in landmarks]
    face_width = max(xs) - min(xs)
    face_height = max(ys) - min(ys)
    return {
        'face_width': face_width,
        'face_height': face_height,
        'face_area': face_width * face_height,
        'face_center_x': (max(xs) + min(xs)) / 2,
        'face_center_y': (max(ys) + min(ys)) / 2
    }


def compute_ear(landmarks):
    def dist(p1, p2):
        return np.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)
    left_ear = (dist(landmarks[160], landmarks[144]) + dist(landmarks[158], landmarks[153])) / \
                (2 * dist(landmarks[33], landmarks[133]))
    right_ear = (dist(landmarks[385], landmarks[380]) + dist(landmarks[387], landmarks[373])) / \
                 (2 * dist(landmarks[362], landmarks[263]))
    return (left_ear + right_ear) / 2


def compute_head_pose(transformation_matrix):
    m = transformation_matrix
    sy = np.sqrt(m[0][0]**2 + m[1][0]**2)
    singular = sy < 1e-6
    if not singular:
        pitch = np.arctan2(m[2][1], m[2][2])
        yaw = np.arctan2(-m[2][0], sy)
        roll = np.arctan2(m[1][0], m[0][0])
    else:
        pitch = np.arctan2(-m[1][2], m[1][1])
        yaw = np.arctan2(-m[2][0], sy)
        roll = 0
    return {'pitch': np.degrees(pitch), 'yaw': np.degrees(yaw), 'roll': np.degrees(roll)}

In [63]:
with open("face_landmarker.task", "rb") as f:
    model_data = f.read()

base_options_v2 = python.BaseOptions(model_asset_buffer=model_data)
options_v2 = vision.FaceLandmarkerOptions(
    base_options=base_options_v2,
    output_face_blendshapes=False,   # is pass mein emotion nahi chahiye, already ho chuka
    output_facial_transformation_matrixes=True,   # head pose ke liye zaroori
    num_faces=1,
    running_mode=vision.RunningMode.IMAGE
)
detector_v2 = vision.FaceLandmarker.create_from_options(options_v2)

In [64]:
def extract_extra_features(video_path, sample_rate_sec=1):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or np.isnan(fps):
        fps = 25
    frame_interval = max(1, int(fps * sample_rate_sec))

    frame_records = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            h, w = frame.shape[:2]
            mp_image = convert_frame_to_mp_image(frame)
            result = detector_v2.detect(mp_image)

            face_geometry = None
            ear = None
            head_pose = None

            if len(result.face_landmarks) > 0:
                landmarks = result.face_landmarks[0]
                face_geometry = compute_face_geometry(landmarks, w, h)
                ear = compute_ear(landmarks)
                if result.facial_transformation_matrixes:
                    head_pose = compute_head_pose(result.facial_transformation_matrixes[0])

            frame_records.append({
                'face_geometry': face_geometry,
                'ear': ear,
                'head_pose': head_pose
            })
        frame_idx += 1

    cap.release()
    return frame_records

In [65]:
def aggregate_extra_features(frame_records, video_id):
    result = {'id': video_id}

    geo_keys = ['face_width', 'face_height', 'face_area', 'face_center_x', 'face_center_y']
    for key in geo_keys:
        vals = [r['face_geometry'][key] for r in frame_records if r['face_geometry'] is not None]
        result[f'{key}_mean'] = np.mean(vals) if vals else np.nan

    ear_vals = [r['ear'] for r in frame_records if r['ear'] is not None]
    result['EAR_mean'] = np.mean(ear_vals) if ear_vals else np.nan
    result['EAR_std'] = np.std(ear_vals) if ear_vals else np.nan

    centers = [(r['face_geometry']['face_center_x'], r['face_geometry']['face_center_y'])
               for r in frame_records if r['face_geometry'] is not None]
    if len(centers) > 1:
        movements = [np.sqrt((centers[i][0]-centers[i-1][0])**2 + (centers[i][1]-centers[i-1][1])**2)
                     for i in range(1, len(centers))]
        result['head_motion_std'] = np.std(movements)
    else:
        result['head_motion_std'] = np.nan

    for angle in ['pitch', 'yaw', 'roll']:
        vals = [r['head_pose'][angle] for r in frame_records if r['head_pose'] is not None]
        result[f'{angle}_mean'] = np.mean(vals) if vals else np.nan
        result[f'{angle}_std'] = np.std(vals) if vals else np.nan

    return result

In [66]:
sample = train_ds[0]
video_path = get_video_path(sample['video'])
frame_records = extract_extra_features(video_path, sample_rate_sec=1)
agg = aggregate_extra_features(frame_records, sample['id'])
print(agg)

{'id': '0001', 'face_width_mean': np.float64(213.91643437472257), 'face_height_mean': np.float64(246.0898086157712), 'face_area_mean': np.float64(52752.95815565931), 'face_center_x_mean': np.float64(344.27194378592753), 'face_center_y_mean': np.float64(278.8277102058584), 'EAR_mean': np.float64(0.29495586635295407), 'EAR_std': np.float64(0.09834532792227856), 'head_motion_std': np.float64(8.008589492678997), 'pitch_mean': np.float64(-12.977782466271094), 'pitch_std': np.float64(4.468693408583492), 'yaw_mean': np.float64(-2.0602380520742254), 'yaw_std': np.float64(3.195966184899133), 'roll_mean': np.float64(-0.775706321466145), 'roll_std': np.float64(2.904071001399732)}


In [68]:
CHECKPOINT_EVERY = 100
results = []
failed_ids = []

for i, sample in enumerate(tqdm(train_ds, desc="Extracting extra features")):
    video_id = sample['id']
    try:
        video_path = get_video_path(sample['video'])
        frame_records = extract_extra_features(video_path, sample_rate_sec=1)
        agg_features = aggregate_extra_features(frame_records, video_id)
        results.append(agg_features)
    except Exception as e:
        print(f"Failed on id={video_id}: {e}")
        failed_ids.append(video_id)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv('face_extra_features_checkpoint.csv', index=False)
        print(f"Checkpoint saved at {i+1} videos")

extra_df = pd.DataFrame(results)
extra_df.to_csv('face_extra_features.csv', index=False)
print("Done:", extra_df.shape)

Extracting extra features:   5%|▍         | 100/2011 [03:46<1:16:03,  2.39s/it]

Checkpoint saved at 100 videos


Extracting extra features:  10%|▉         | 200/2011 [06:53<56:57,  1.89s/it]  

Checkpoint saved at 200 videos


Extracting extra features:  15%|█▍        | 300/2011 [11:17<1:13:23,  2.57s/it] 

...

Checkpoint saved at 1900 videos


Extracting extra features:  99%|█████████▉| 2000/2011 [1:17:57<00:24,  2.20s/it]

Checkpoint saved at 2000 videos


Extracting extra features: 100%|██████████| 2011/2011 [1:18:21<00:00,  2.34s/it]

Done: (2011, 15)


In [69]:
main_df = pd.read_csv('face_features_final.csv')    
extra_df = pd.read_csv('face_extra_features.csv')     
merged_df = main_df.merge(extra_df, on='id', how='inner')
merged_df.to_csv('face_features_complete.csv', index=False)
print(merged_df.shape)   

(2011, 38)


In [70]:
df = pd.read_csv("face_features_complete.csv")

In [72]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2011 entries, 0 to 2010
Data columns (total 38 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     2011 non-null   int64  
 1   face_detected_ratio    2011 non-null   float64
 2   gaze_ratio_mean        2011 non-null   float64
 3   gaze_deviation_mean    2011 non-null   float64
 4   gaze_stability_std     2011 non-null   float64
 5   smile_score_mean       2011 non-null   float64
 6   smile_score_std        2011 non-null   float64
 7   frown_score_mean       2011 non-null   float64
 8   frown_score_std        2011 non-null   float64
 9   eye_openness_mean      2011 non-null   float64
 10  eye_openness_std       2011 non-null   float64
 11  jaw_open_mean          2011 non-null   float64
 12  jaw_open_std           2011 non-null   float64
 13  brow_raise_mean        2011 non-null   float64
 14  brow_raise_std         2011 non-null   float64
 15  mouth_frown_mea